# Model Testing

Implement model and testing system.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))
import torch
from src.dataset import encode_text, decode_tokens
from src.model import CausalLanguageModel
from src.inference import generate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = "../checkpoints/model.pt"
checkpoint = torch.load(ckpt_path, map_location=device, weights_only=True)
display("Tokenizer:", checkpoint["tokenizer"])

model = CausalLanguageModel(**checkpoint["model_config"]).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
display("Model loaded:", checkpoint["model_config"])
display(model)

'Tokenizer:'

'utf8_bytes'

d:\VennJr\alpha-torino\.venv\lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


'Model loaded:'

{'vocab_size': 256,
 'block_size': 128,
 'd_model': 128,
 'n_heads': 4,
 'n_layers': 4,
 'dropout': 0.1}

CausalLanguageModel(
  (token_embedding): Embedding(256, 128)
  (position_embedding): Embedding(128, 128)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (final_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_features=128, out_features=256, bias=False)
)

In [5]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total of Parametors: {total_params:,}")
print(f"Trainable parametors: {trainable_params:,}")
print(f"Size ~{total_params * 4 / 1024**2:.1f} MB (float32)")

Total of Parametors: 875,264
Trainable parametors: 875,264
Size ~3.3 MB (float32)


In [22]:
prompt = "Don't do that"
prompt_ids = torch.tensor([encode_text(prompt)], dtype=torch.long, device=device)

output = generate(model, prompt_ids, max_new_tokens=100, temperature=0.8)
print(decode_tokens(output[0]))

Don't do that." Don't make the flower and the hungry sad.

But then, a flower came into the attic and saw the t


In [25]:
ids = torch.randint(0, checkpoint["model_config"]["vocab_size"], (2, 32), device=device)
logits, loss = model(ids, targets=ids)
print("Logits shape:", logits.shape)   # (2, 32, vocab_size)
print("Loss:", loss.item())


Logits shape: torch.Size([2, 32, 256])
Loss: 19.343181610107422
